# DLP Random Event Generator
Generates realistic synthetic events and appends them to the Delta table.
Scheduled to run daily via Databricks Workflows.
Produces between 10-50 new events per run with realistic noise and user behaviour patterns.

In [0]:
import os
import csv
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F

## 1. Configuration

In [0]:
TARGET_TABLE = "workspace.default.cloud_app_events_banking_dlp_v_2"

# Number of new events to generate per run
MIN_EVENTS = 10
MAX_EVENTS = 50

# Seed changes daily so each run generates different events
SEED = int(datetime.now().strftime("%Y%m%d"))
random.seed(SEED)
np.random.seed(SEED)

print(f"Run date : {datetime.now().strftime('%Y-%m-%d')}")
print(f"Seed     : {SEED}")
print(f"Target   : {TARGET_TABLE}")

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

## 2. Load Reference Data
Loads risky domains, safe/risky filenames from CSVs in the repo root.
Keeps reference data maintainable without touching notebook code.

In [0]:
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = os.path.dirname(os.path.dirname(notebook_path))

# Risky domains
with open(f"/Workspace{repo_root}/reference_data/risky_domains.csv") as f:
    risky_domains = [row["Domain"].strip() for row in csv.DictReader(f)]

# Safe filenames
with open(f"/Workspace{repo_root}/reference_data/safe_filenames.csv") as f:
    safe_filenames = [row["filename"].strip() for row in csv.DictReader(f)]

# Risky filenames
with open(f"/Workspace{repo_root}/reference_data/risky_filenames.csv") as f:
    risky_filenames = [row["filename"].strip() for row in csv.DictReader(f)]

safe_domains = [
    "teams.microsoft.com", "outlook.office365.com", "workday.com",
    "secure.bank.com", "internal.bank.sharepoint.com"
]

positions = [
    "Software Developer", "Loan Officer", "Relationship Banker",
    "System Architect", "Wealth Manager"
]

print(f"Risky domains   : {len(risky_domains)}")
print(f"Safe domains    : {len(safe_domains)}")
print(f"Safe filenames  : {len(safe_filenames)}")
print(f"Risky filenames : {len(risky_filenames)}")

## 3. Get Current Max Action_ID
Ensures new events don't overlap with existing IDs.

In [0]:
existing_df = spark.table(TARGET_TABLE)
max_id  = existing_df.agg(F.max("Action_ID")).collect()[0][0]
next_id = max_id + 1

print(f"Existing rows  : {existing_df.count()}")
print(f"Max Action_ID  : {max_id}")
print(f"Next Action_ID : {next_id}")

## 4. Generate New Events
Labels are assigned based on a weighted risk score combining domain,
file, time, and position signals with added noise — same logic as the
original mock data generator to ensure consistent label distribution.

In [0]:
n_events = random.randint(MIN_EVENTS, MAX_EVENTS)
now      = datetime.utcnow()
start_ts = now - timedelta(hours=24)

rows      = []
action_id = next_id

for _ in range(n_events):
    user     = f"User_{random.randint(1, 80):03d}"
    position = random.choice(positions)

    # Timestamp within last 24 hours
    ts   = start_ts + timedelta(seconds=random.randint(0, 86400))
    hour = ts.hour

    # ── Risk score ────────────────────────────────────────────────────────────
    risk_score = 0.0

    # Domain
    if random.random() < 0.15:
        domain      = random.choice(risky_domains)
        risk_score += 0.35
    else:
        domain      = random.choice(safe_domains)
        risk_score -= 0.1

    # File
    if random.random() < 0.3:
        filename    = random.choice(risky_filenames)
        risk_score += 0.30
    else:
        filename    = random.choice(safe_filenames)
        risk_score -= 0.05

    # Time
    if hour < 8 or hour >= 19:
        risk_score += 0.15

    # Position
    if position in ["Wealth Manager", "Loan Officer"]:
        risk_score += 0.08

    # Day of week
    if ts.weekday() in [0, 4]:  # Monday or Friday
        risk_score += 0.05

    # Noise
    risk_score += np.random.normal(0, 0.15)
    label = 1 if risk_score > 0.35 else 0

    rows.append({
        "Action_ID":          action_id,
        "Timestamp":          ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "ActionType":         "FileUploaded",
        "ObjectName":         filename,
        "Target_Domain":      domain,
        "AccountDisplayName": user,
        "Position":           position,
        "Risk_Label":         label
    })
    action_id += 1

new_df = pd.DataFrame(rows)
print(f"Generated {n_events} new events")
print(f"Label distribution:\n{new_df['Risk_Label'].value_counts()}")
print(new_df[["Action_ID", "Timestamp", "ObjectName", "Target_Domain", "Risk_Label"]].to_string(index=False))

## 5. Deduplicate and Append
Casts Timestamp to match Delta table schema, deduplicates against
existing Action_IDs, then appends to the Delta table.

In [0]:
new_spark_df = spark.createDataFrame(new_df)

# Cast Timestamp string to timestamp type to match Delta table schema
new_spark_df = new_spark_df.withColumn(
    "Timestamp",
    F.to_timestamp(F.col("Timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
)

# Deduplicate against existing IDs
existing_ids = existing_df.select("Action_ID")
new_spark_df = new_spark_df.join(existing_ids, on="Action_ID", how="left_anti")

rows_to_append = new_spark_df.count()
print(f"Rows to append after deduplication: {rows_to_append}")

# Append to Delta table
new_spark_df.write.mode("append").saveAsTable(TARGET_TABLE)

# Verify
total_rows = spark.table(TARGET_TABLE).count()
print(f"Total rows in table after append: {total_rows}")

In [0]:

today = datetime.utcnow().strftime("%Y-%m-%d")

display(
    spark.table(TARGET_TABLE)
    .filter(F.col("Timestamp").cast("date") == today)
    .orderBy("Action_ID")
)